# 第5章 モータ異音検知（CWT／STFT・異常検知・ROC）（ノートブック版）

教科書の第5章の 5-6 のプログラムのうち、**合成のモータ音で動く解析の部分**を、本文の `setup_ch5.sh`・`run_ch5.sh` の流れのとおりに1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約1分。Colab ではその数倍かかることがあります）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | 合成の「録音のセッション」19 個を作り、音を聞く | A の 1 | `synth_sound.py` |
| 2 | 時刻をそろえて 4096 点ずつに切り、データセットにする | A の 1・E の 9 | `ch5_dataset.py` |
| 3 | スペクトルの線をエンコーダの回転数と照らし合わせる（PWM の線・時計のずれ・側帯波） | C の 6・E の 10 | `lines.py` |
| 4 | STFT（短い窓・長い窓）・CWT・対数メルを並べて比べ、PyWavelets と比べる | E の 11 | `tf_compare.py` |
| 5 | 正常だけで学ぶ異常検知（ガウス＋マハラノビス、オートエンコーダ）と ROC-AUC、区間 | F の 12・13 | `detect.py` |
| 6 | 音から回転数を推定し、ブランド・アルトマン図で比べる | F の 14 | `rpm_audio.py` |
| 7 | III-6 の課題1 の入口（`MotorRec` の数と、`session_ci.py` のセッション単位の区間） | G の 15（一部） | `motor_rec.py`・`session_ci.py` |

- 次のセルたちが、5-6 のファイルを、このノートブックの隣に `%%writefile` で書き出します（教科書の `setup_ch5.sh` で `~/home-lab/ch5/` に保存する代わり）。中身は 5-6 と同じです。変えたのは、図の日本語フォントの候補を足したこと（`ch5lib.py` の `jp_plt` の `plt.rcParams["font.family"]` の行）だけです。
- 各節では、`run_ch5.sh` の `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します（中では `python ○○.py …` を別のプロセスで動かします）。録音のフォルダ `data/sessions` は、合成データの `data_synth/sessions` に、図の置き場所 `out/` は `out_synth/` に置き換えています。表示を見てから、`ok()` で教科書の手順と 5-7 の「結果の例」を確かめます。
- 計算はどれも軽く、オートエンコーダ（40 エポック × 種 3 つ）とブートストラップ（2000 回）も教科書と同じ設定のままです（執筆時の Mac で `detect.py` 1回 約10秒。Colab の CPU でもそれほどかかりません）。重いと感じたら、5 節のセルの `SEEDS` と `EPOCHS` を小さくしてください（そのときは AE の確かめは「種ごとの値」ではなく、AUC がおおよその範囲にあるかだけを見ます）。
- **入れていないもの（Colab ではできない手順）**：Mac の内蔵マイクと Pico B（第4章の motor の UF2）を使った録音 `rec_session.py`（`devices`・`level`・`plan`・`record`。手順 B〜D）と、それが使う第4章の `motor_run.py`、テスタでの確認。録音は Mac で教科書の手順のとおりに行ってください。録った `data/sessions/` を Colab の左のファイルの欄に上げれば、2〜6 節の `data_synth/sessions` を `data/sessions` に、`--cpr-out`・`--pulses` を自分の値に書きかえて同じ解析ができるはずですが、執筆時には確かめていません。III-6 の `anomaly_ae.py`（変更点を当てたもの）の学習も入れていません（7 節は、その入口だけです）。
- ⚠ 音を聞くセル（1 節）は、**先にスピーカー・イヤホンの音量を下げてから**実行してください。セルは、録ったままの大きさ（`normalize=False`）で鳴らし、最大が −6 dBFS を超える音は鳴らしません。
- 数値は乱数の種を変えなければ教科書と同じですが、numpy・scipy・librosa・PyWavelets・torch の版や、オートエンコーダを動かす機器（Mac の MPS・CPU・Colab の GPU）で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。合成データの値は仮の値で、実機の値ではありません。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・scikit-learn・torch・librosa・soundfile が
#    最初から入っていることが多い。PyWavelets は入っていないことがある。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("pywt", "PyWavelets"), ("librosa", "librosa"), ("soundfile", "soundfile"),
                 ("sklearn", "scikit-learn"), ("torch", "torch"), ("pandas", "pandas"), ("scipy", "scipy")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, librosa, sklearn, torch, pywt, soundfile
print("OK", "numpy", numpy.__version__, "scipy", scipy.__version__, "librosa", librosa.__version__,
      "scikit-learn", sklearn.__version__, "torch", torch.__version__, "PyWavelets", pywt.__version__)
print("GPU（CUDA）", torch.cuda.is_available(), " MPS", torch.backends.mps.is_available())
print("（教科書は librosa 1.0.1・scikit-learn 1.9.1・torch 2.15.0 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の8つのセルで、5-6 のファイルを書き出します（`ch5lib.py` は共通の関数で、ほかの7つが読み込みます）。録音の `rec_session.py` は書き出しません。

In [ ]:
%%writefile ch5lib.py
#!/usr/bin/env python3
"""第5章：ほかのスクリプトが import する共通の関数（~/home-lab/ch5/ch5lib.py）

  load_run(sdir, cond)              録音（wav）・エンコーダ（CSV）・記録の情報（JSON）を読む
  pico_offset(enc)                  Pico の時刻 t_ms と Mac の時刻（time.monotonic）のずれ（最小遅延の推定）
  run_start_sample(meta, enc, fs)   RUN の開始（t_ms = 0）が、録音の何番目の標本に当たるか（時計から）
  onset_sample(x, fs, n_guess)      音の立ち上がりの標本（時計による時刻合わせの確かめ用）
  to_16k(x, fs)                     16 kHz にする（III-2 の SynthMotor・III-6 と同じ標本化周波数）
  run_end_s(enc, meta)              指令が出ていた最後の時刻 [s]（途中で止まったら短くなる）
  encoder_rate(enc, t0_ms, t1_ms)   区間のカウントの増え方 [カウント/s]（最小二乗の傾き）
  logmel(x, fs)                     対数メルスペクトログラム [dB]、形 (n_mels, フレーム数)
  cwt_morlet(x, fs, freqs)          複素モルレーの連続ウェーブレット変換の大きさ（FFT で計算）
  harmonic_f0(x, fs, fmin, fmax)    倍音の和がいちばん大きい基本周波数（音から回転数を推定）
  roc_auc・roc_curve・boot_auc     ROC-AUC と、ブートストラップの 95% 区間（区間単位・セッション単位）
"""
import json
import pathlib
from fractions import Fraction

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.ndimage import median_filter
from scipy.signal import butter, resample_poly, sosfiltfilt
from scipy.stats import rankdata

FS_DS = 16000                 # データセットの標本化周波数（III-6 の SynthMotor と同じ）
SEG = 4096                    # 1区間の長さ（16 kHz で 0.256 秒）
KINDS = {"N": "正常", "U": "アンバランス", "F": "摩擦", "V": "電圧低下"}
CONDS = {"S1": (1000, 1000.0), "S2": (700, 1000.0)}   # 第4章の標準の運転条件：（デューティ‰、PWM の周波数 Hz）


def kind_of(session):
    """セッションの名前の1文字目が種類（N01 → N）"""
    return session[0]


def list_sessions(root):
    return sorted(p.name for p in pathlib.Path(root).iterdir() if p.is_dir() and p.name[0] in KINDS)


def load_run(sdir, cond):
    sdir = pathlib.Path(sdir)
    x, fs = sf.read(str(sdir / f"{cond}.wav"), dtype="float64", always_2d=False)
    if x.ndim == 2:
        x = x[:, 0]
    enc = pd.read_csv(sdir / f"{cond}_enc.csv")
    meta = json.loads((sdir / f"{cond}.json").read_text())
    return x, int(fs), enc, meta


def pico_offset(enc):
    """mono = off + t_ms/1000 + 遅れ（遅れ ≥ 0）。遅れのいちばん小さい行がいちばん正しいので、最小値を使う。
    返り値：(off [s], 遅れの中央値 [ms], 遅れの 99% 点 [ms])"""
    d = enc["mono"].to_numpy() - enc["t_ms"].to_numpy() / 1000.0
    off = float(d.min())
    lag = (d - off) * 1000
    return off, float(np.median(lag)), float(np.percentile(lag, 99))


def run_start_sample(meta, enc, fs):
    """t_ms = 0 の時刻が、録音の何番目の標本か（小数）。audio_t0_mono は録音の最初の標本の Mac の時刻"""
    off, _, _ = pico_offset(enc)
    return (off - meta["audio_t0_mono"]) * fs


def onset_sample(x, fs, n_guess, search_s=0.3, frame_s=0.002, rise_db=10.0):
    """n_guess の前後 search_s 秒で、1 kHz より上の音の大きさ（2 ms ごと）が、その前の静かな 0.5 秒より
    rise_db 上がった最初の点（ブラシの雑音は高い音まで出るので、部屋の低い音を除いて探す）"""
    n = int(frame_s * fs)
    a = int(n_guess - search_s * fs)                          # 探し始める標本
    b = min(len(x), int(n_guess + search_s * fs))
    c = a - int(0.5 * fs)                                     # 静かな部分の始まり（モータは止まっている）
    if c < 0 or b - a < 10 * n:
        return float("nan")
    h = sosfiltfilt(butter(4, 1000, btype="highpass", fs=fs, output="sos"), x[c:b])

    def frames_db(v):
        m = len(v) // n
        return 10 * np.log10(np.mean(v[:m * n].reshape(m, n) ** 2, axis=1) + 1e-20)

    base = np.median(frames_db(h[:a - c]))
    above = frames_db(h[a - c:]) > base + rise_db
    for i in range(len(above) - 3):
        if above[i:i + 4].all():                              # 4 フレーム続けて大きい（1回だけの雑音で止まらない）
            return float(a + i * n)
    return float("nan")


def to_16k(x, fs):
    r = Fraction(FS_DS, int(fs)).limit_denominator(1000)
    return x if r == 1 else resample_poly(x, r.numerator, r.denominator)


def run_end_s(enc, meta):
    """モータに指令が出ていた最後の時刻 [s]（RUN の開始から）。途中で止まった（STOP・I,undervolt）ときは指令より短い"""
    on = enc.loc[enc["duty"] > 0, "t_ms"]
    return min(meta["ms1"], float(on.max()) + 1.0) / 1000 if len(on) else 0.0


def encoder_rate(enc, t0_ms, t1_ms):
    sel = (enc["t_ms"] >= t0_ms) & (enc["t_ms"] < t1_ms)
    t = enc.loc[sel, "t_ms"].to_numpy() / 1000.0
    c = enc.loc[sel, "count"].to_numpy(dtype=float)
    if len(t) < 10:
        return float("nan")
    return float(np.polyfit(t, c, 1)[0])


def logmel(x, fs, n_fft=1024, hop=256, n_mels=64, fmin=50.0, fmax=None):
    import librosa
    m = librosa.feature.melspectrogram(y=np.asarray(x, float), sr=fs, n_fft=n_fft, hop_length=hop,
                                       n_mels=n_mels, fmin=fmin, fmax=fmax, power=2.0)
    return 10 * np.log10(m + 1e-10)


def cwt_morlet(x, fs, freqs, n_cycles=6.0, hop=1):
    """複素モルレー：時間の広がり σ_t = n_cycles/(2πf)、周波数の広がり σ_f = f/n_cycles。
    周波数の側でガウスの山（正の周波数だけ、高さ 2）を掛けて逆 FFT するので、振幅 A の正弦波は |W| ≈ A になる。
    返り値：(len(freqs), 長さ/hop) の大きさ"""
    x = np.asarray(x, float)
    pad = int(4 * n_cycles / (2 * np.pi * min(freqs)) * fs)  # いちばん低い周波数の ±4σ_t ぶん、両端にゼロ
    nfft = 1 << int(np.ceil(np.log2(len(x) + 2 * pad)))
    X = np.fft.rfft(np.concatenate([np.zeros(pad), x]), nfft)
    nu = np.fft.rfftfreq(nfft, 1 / fs)
    out = np.empty((len(freqs), (len(x) + hop - 1) // hop), dtype=np.float32)
    for i, f in enumerate(freqs):
        H = 2.0 * np.exp(-0.5 * ((nu - f) / (f / n_cycles)) ** 2)
        full = np.zeros(nfft, complex)
        full[:len(nu)] = X * H                                 # 解析信号（負の周波数は 0）にする
        w = np.fft.ifft(full)[pad:pad + len(x)]
        out[i] = np.abs(w[::hop])
    return out


def harmonic_f0(x, fs, fmin, fmax, n_harm=10, res_hz=0.25, df=0.05):
    """モータ軸の回転の周波数 f を、k·f（k = 1…n_harm）の線の強さの和が最大になる f として探す。
    線の強さは、まわり 100 Hz の中央値（雑音の床）から何 dB 上か（0〜30 dB に切る）。
    返り値：(f [Hz], 和の値, 2番目の山との比)"""
    x = np.asarray(x, float) - np.mean(x)
    nfft = max(len(x), int(round(fs / res_hz)))
    P = np.abs(np.fft.rfft(x * np.hanning(len(x)), nfft)) ** 2
    fr = np.arange(len(P)) * fs / nfft
    L = 10 * np.log10(P + 1e-20)
    floor = median_filter(L, size=int(100 / (fs / nfft)) | 1, mode="nearest")
    E = np.clip(L - floor, 0.0, 30.0)
    cand = np.arange(fmin, fmax, df)
    score = np.zeros_like(cand)
    for k in range(1, n_harm + 1):
        ok = k * cand < 0.95 * fs / 2
        score[ok] += np.interp(k * cand[ok], fr, E)
    i = int(np.argmax(score))
    if 0 < i < len(cand) - 1:                                  # 放物線で山の頂上を細かく
        a, b, c = score[i - 1], score[i], score[i + 1]
        den = a - 2 * b + c
        f = cand[i] + (0.5 * (a - c) / den * df if den < 0 else 0.0)
    else:
        f = cand[i]
    far = np.abs(cand - cand[i]) > 0.1 * cand[i]               # ±10% より外の、いちばん高い山
    second = score[far].max() if far.any() else 0.0
    return float(f), float(score[i]), float(score[i] / max(second, 1e-9))


def roc_auc(s0, s1):
    """AUC = P(異常の得点 > 正常の得点)。マン・ホイットニーの U を n0·n1 で割ったもの（同点は 1/2）"""
    s0, s1 = np.asarray(s0, float), np.asarray(s1, float)
    r = rankdata(np.concatenate([s0, s1]))
    n0, n1 = len(s0), len(s1)
    return float((r[n0:].sum() - n1 * (n1 + 1) / 2) / (n0 * n1))


def roc_curve(s0, s1):
    """しきい値を高いほうから下げたときの（偽陽性率, 真陽性率）"""
    th = np.sort(np.unique(np.concatenate([s0, s1])))[::-1]
    fpr = np.array([0.0] + [np.mean(s0 >= t) for t in th])
    tpr = np.array([0.0] + [np.mean(s1 >= t) for t in th])
    return fpr, tpr


def boot_auc(s0, s1, g0=None, g1=None, n_boot=2000, seed=0):
    """AUC の 95% ブートストラップ区間。
    g0・g1 が None：区間（セグメント）を1つずつ復元抽出（区間どうしが独立という仮定。III-6 の auc_boot_ci と同じ）
    g0・g1 にセッションの名前：セッションを復元抽出し、選んだセッションの区間を全部使う（クラスタ・ブートストラップ）"""
    rng = np.random.default_rng(seed)
    s0, s1 = np.asarray(s0, float), np.asarray(s1, float)
    out = []
    if g0 is None:
        for _ in range(n_boot):
            out.append(roc_auc(rng.choice(s0, len(s0)), rng.choice(s1, len(s1))))
    else:
        g0, g1 = np.asarray(g0), np.asarray(g1)
        u0, u1 = np.unique(g0), np.unique(g1)
        idx0 = {g: np.flatnonzero(g0 == g) for g in u0}
        idx1 = {g: np.flatnonzero(g1 == g) for g in u1}
        for _ in range(n_boot):
            a = np.concatenate([idx0[g] for g in rng.choice(u0, len(u0))])
            b = np.concatenate([idx1[g] for g in rng.choice(u1, len(u1))])
            out.append(roc_auc(s0[a], s1[b]))
    lo, hi = np.percentile(out, [2.5, 97.5])
    return float(lo), float(hi)


def jp_plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
    plt.rcParams["axes.unicode_minus"] = False
    return plt

In [ ]:
%%writefile synth_sound.py
#!/usr/bin/env python3
"""第5章：解析を試すための合成の「録音のセッション」を作る（実機がなくても、解析の流れを最後まで確かめられる）

使い方（~/home-lab/ch5 で）:
  uv run python synth_sound.py --root data_synth/sessions --seed 0
rec_session.py record と同じ形のファイル（<セッション>/S1.wav・S1_enc.csv・S1.json、S2 も）を、
正常 N01〜N10、アンバランス U01〜U03、摩擦 F01〜F03、電圧低下 V01〜V03 の 19 セッションぶん作る。

合成の中身（実機の値ではなく、仮の値）：
  P = 7、G = 98（CPR_out = 2744。第4章の synth_motor.py と同じ）、電池 5.0〜5.4 V（セッションごと）。
  S2（70%・1 kHz）の回転は S1 の 0.9 倍とした（無負荷の PWM は電流が途切れ、デューティに比例しない。実機はエンコーダで測る）。
  INA219 の電流は電池側の電流（PWM ではおよそ デューティ×モータの電流）。
  音 = モータ軸の回転の周波数 f_m の倍音（k = 1〜12。k = 6 は整流子、k = 10 はギアのかみ合いとして強め）
       ＋ブラシの雑音（1.5〜7 kHz、6·f_m で脈打つ）＋（S2 だけ）PWM 1 kHz とその倍音＋部屋の雑音。
  異常：U＝出力軸1回転ごとの振幅の変化（変調の深さ 0.25〜0.4）、F＝負荷で遅くなり、こすれる音（2.5〜6 kHz）、
        V＝電圧が約 0.65 V 下がって遅く・小さくなる。セッションごとに置き方（音量 ±1.5 dB、線の強さ）が変わる。
  時計：Mac の録音の時計は Pico より 30 ppm 速い。録音の最初の標本の時刻には 3 ms 前後の誤差を入れる。
"""
import argparse
import csv
import datetime
import json
import pathlib

import numpy as np
import soundfile as sf
from scipy.signal import butter, sosfilt

P_TRUE, G_TRUE = 7, 98
CPR_OUT = 4 * P_TRUE * G_TRUE
FS = 48000
TAU, TAU_STOP = 0.040, 0.060                    # 回り始めと止まるときの時定数 [s]
R_OHM, KE, I_F = 8.0, 0.161, 0.060              # 第4章の synth_motor.py と同じ
EPS_MAC = 30e-6                                 # Mac の録音の時計の速さのずれ（Pico に対して）
FIELDS = ["t_ms", "duty", "count", "ina_seq", "i_ma", "v_bus", "host_time", "mono"]
CONDS = {"S1": (1000, 1000.0), "S2": (700, 1000.0)}


def ou(rng, n, dt, sd, tc):
    """平均 0・標準偏差 sd・相関時間 tc のゆらぎ（オルンシュタイン–ウーレンベック過程）"""
    a = np.exp(-dt / tc)
    e = rng.normal(0, sd * np.sqrt(1 - a * a), n)
    y = np.empty(n)
    y[0] = rng.normal(0, sd)
    for i in range(1, n):
        y[i] = a * y[i - 1] + e[i]
    return y


def band_noise(rng, n, lo, hi):
    sos = butter(4, [lo, hi], btype="bandpass", fs=FS, output="sos")
    y = sosfilt(sos, rng.standard_normal(n))
    return y / y.std()


def pink(rng, n):
    X = np.fft.rfft(rng.standard_normal(n))
    f = np.fft.rfftfreq(n, 1 / FS)
    X[1:] /= np.sqrt(f[1:] / 100.0)
    X[0] = 0
    y = np.fft.irfft(X, n)
    return y / y.std()


def make_run(rng, kind, cond, sess, ms1=10000, pre_s=2.0, post_s=0.8):
    d1, f_pwm = CONDS[cond]
    u = d1 / 1000
    v_batt = sess["v_batt"] - (0.65 if kind == "V" else 0.0)
    i_load = 0.045 if kind == "F" else 0.0
    i_nl = I_F + i_load
    w_ss = (v_batt - R_OHM * i_nl) / KE                        # デューティ 100% の定常の角速度 [rad/s]（出力軸）
    if u < 1:
        w_ss *= 0.9                                            # 無負荷の PWM は電流が途切れ、デューティに比例しない（第4章：S1 の9割前後）
    f_ss = w_ss / (2 * np.pi)                                  # 出力軸の回転の周波数 [Hz]
    T = ms1 / 1000
    # ---- 「本当の」時刻（RUN の開始が 0）。録音の標本 n の時刻は t_a0 + n/(FS(1+ε)) ----
    t_a0 = -pre_s - rng.uniform(0.0, 0.05)
    n = int((pre_s + T + 0.3 + post_s) * FS)
    t = t_a0 + np.arange(n) / (FS * (1 + EPS_MAC))
    t_snd = t - 0.2 / 343.0                                    # 20 cm を音が進む時間だけ遅れて届く
    s = np.where(t_snd < 0, 0.0, np.where(t_snd < T, 1 - np.exp(-np.clip(t_snd, 0, None) / TAU),
                                          (1 - np.exp(-T / TAU)) * np.exp(-(t_snd - T) / TAU_STOP)))
    wob = ou(rng, n // 480 + 2, 0.01, 0.003, 0.5)              # 回転のゆらぎ 0.3%（10 ms ごとに作って補間）
    wob = np.interp(np.arange(n) / 480, np.arange(len(wob)), wob)
    f_out = f_ss * s * (1 + wob)
    rev_out = np.cumsum(f_out) / (FS * (1 + EPS_MAC))          # 出力軸の回転数（積分）
    ph_m = 2 * np.pi * G_TRUE * rev_out                        # モータ軸の位相
    ph_o = 2 * np.pi * rev_out + sess["ph_o"]
    i_t = np.where((t_snd >= 0) & (t_snd < T), i_nl + 0.5 * np.exp(-np.clip(t_snd, 0, None) / TAU), 0.0)
    # ---- 音 ----
    am = 1 + 0.08 * np.cos(ph_o)                               # 正常でも、出力軸の少しの偏心でかみ合いの音がゆれる
    am_all = np.ones(n)
    if kind == "U":
        am_all = 1 + sess["m_u"] * np.cos(ph_o)                # アンバランス：出力軸1回転ごとに全体の振幅が変わる
        am = am * (1 + 0.5 * sess["m_u"] * np.cos(ph_o + 0.7))
    mot = np.zeros(n)
    for k in range(1, 13):
        a = 0.010 * k ** -0.7 * sess["harm"][k - 1]
        if k == 6:
            a += 0.012 * sess["comm"] * (1.2 if kind == "F" else 1.0)
        if k == 10:
            a = (a + 0.010 * sess["mesh"]) * am
        mot += a * np.sin(k * ph_m + sess["phase"][k - 1])
    brush = 0.006 * sess["brush"] * band_noise(rng, n, 1500, 7000) * ((1 + np.cos(6 * ph_m)) / 2) ** 2
    brush *= np.sqrt(i_t / I_F)
    mot = (mot * s + brush) * am_all
    if kind == "V":
        mot *= 0.85
    if kind == "F":
        mot += 0.004 * sess["rub"] * band_noise(rng, n, 2500, 6000) * (0.6 + 0.4 * np.cos(ph_o)) * (s > 0.5)
    if cond == "S2":
        on = (t_snd >= 0) & (t_snd < T)
        for m in range(1, 9):
            mot += on * 0.006 / m * np.sin(2 * np.pi * m * f_pwm * t + m)
    room = 10 ** (sess["bg_db"] / 20) * pink(rng, n)
    x = 10 ** (sess["gain_db"] / 20) * mot + room
    # ---- エンコーダ（Pico の 1 ms ごとの記録）----
    rows = []
    t_ms = np.arange(0, ms1 + 301)
    rev_ms = np.interp(t_ms / 1000, t, rev_out)
    count = np.floor(CPR_OUT * rev_ms).astype(int)
    i_ms = u * np.interp(t_ms / 1000, t, i_t)                  # INA219 は電池側：PWM ではおよそ デューティ×モータの電流
    m0 = sess["mono0"]
    delay = 0.0008 + rng.exponential(0.003, len(t_ms))
    delay[rng.random(len(t_ms)) < 0.01] += rng.uniform(0.01, 0.03)   # ときどき USB の受け取りが遅れる
    seq, i_avg = 0, 0.0
    for k, tm in enumerate(t_ms):
        if tm % 9 == 0:                                       # INA219 の 16 回平均（約 8.5 ms）ごとに新しい値
            seq += 1
            i_avg = i_ms[max(0, k - 8):k + 1].mean() * 1000 + rng.normal(0, 1.0)
        mono = m0 + tm / 1000 + delay[k]
        rows.append({"t_ms": int(tm), "duty": d1 if tm < ms1 else 0, "count": int(count[k]), "ina_seq": seq,
                     "i_ma": f"{i_avg:.1f}", "v_bus": f"{v_batt - 0.5 * i_avg / 1000:.3f}",
                     "host_time": f"{1.7e9 + mono:.3f}", "mono": f"{mono:.6f}"})
    lat_err = rng.normal(0.003, 0.001)                         # 録音の時刻の印の誤差（Mac の側で分からない遅れ）
    meta = {"session": sess["name"], "kind": kind, "cond": cond, "d1": d1, "ms1": ms1, "freq_actual": f_pwm,
            "fs": FS, "device": "合成（synth_sound.py）", "pre_s": pre_s,
            "audio_t0_mono": m0 + t_a0 + lat_err, "input_volume": None, "distance_cm": 20,
            "i_max_ma": 400, "dropped": 0, "stopped": False, "overflow": [],
            "date": datetime.datetime.now().isoformat(timespec="seconds"),
            "truth": {"f_out_ss": f_ss, "f_m_ss": f_ss * G_TRUE, "P": P_TRUE, "G": G_TRUE,
                      "eps_mac_ppm": EPS_MAC * 1e6, "lat_err_ms": lat_err * 1000, "v_batt": v_batt}}
    return x, rows, meta


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="data_synth/sessions")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ms", type=int, default=10000)
    a = ap.parse_args()
    root = pathlib.Path(a.root)
    rng = np.random.default_rng(a.seed)
    harm0 = np.exp(rng.normal(0, 0.4, 12))                     # このモータ（1個）の倍音の強さのくせ
    names = [f"N{i:02d}" for i in range(1, 11)] + [f"{k}{i:02d}" for k in "UFV" for i in range(1, 4)]
    for j, name in enumerate(names):
        kind = name[0]
        sess = {"name": name, "v_batt": rng.uniform(5.0, 5.4), "gain_db": rng.normal(0, 1.5),
                "bg_db": -58 + rng.normal(0, 1.0), "harm": harm0 * np.exp(rng.normal(0, 0.15, 12)),
                "comm": np.exp(rng.normal(0, 0.15)), "mesh": np.exp(rng.normal(0, 0.2)),
                "brush": np.exp(rng.normal(0, 0.15)), "rub": np.exp(rng.normal(0, 0.2)),
                "phase": rng.uniform(0, 2 * np.pi, 12), "ph_o": rng.uniform(0, 2 * np.pi),
                "m_u": rng.uniform(0.25, 0.4), "mono0": 5000.0 + 300 * j}
        d = root / name
        d.mkdir(parents=True, exist_ok=True)
        for cond in CONDS:
            x, rows, meta = make_run(rng, kind, cond, sess, ms1=a.ms)
            sess["mono0"] += 30.0
            peak = np.max(np.abs(x))
            meta["peak_dbfs"] = float(20 * np.log10(peak))
            sf.write(str(d / f"{cond}.wav"), x, FS, subtype="PCM_24")
            with open(d / f"{cond}_enc.csv", "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=FIELDS)
                w.writeheader()
                w.writerows(rows)
            (d / f"{cond}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))
        print(f"{name}：{KIND_JA[kind]}、電池 {sess['v_batt']:.2f} V、最大 {meta['peak_dbfs']:.1f} dBFS")
    print(f"{len(names)} セッションを {root} に作りました")


KIND_JA = {"N": "正常", "U": "アンバランス", "F": "摩擦", "V": "電圧低下"}

if __name__ == "__main__":
    main()

In [ ]:
%%writefile ch5_dataset.py
#!/usr/bin/env python3
"""第5章：録音したセッションを、時刻をそろえてから 4096 点（16 kHz で 0.256 秒）ずつに切り、データセットにする

使い方（~/home-lab/ch5 で）:
  uv run python ch5_dataset.py --root data/sessions --cpr-out 2744 --pulses 7 --out data/ch5_dataset
  --cpr-out：第4章の手順6 で測った N_out（出力軸1回転のカウント）。--pulses：P（分からなければ省く）
  --align clock（既定：Pico と Mac と録音の時計から）｜onset（音の立ち上がりから）
  --n-train 6 --seed 0：学習に使う正常セッションの数と、選ぶ乱数の種（セッション単位で分ける）

出力（条件 S1・S2 ごとのフォルダ）:
  x.npy (N, 4096) float32（16 kHz、録音のままの大きさ）、y.npy (N,) int64（0＝正常、1＝異常）、
  session.npy (N,) 文字列、segments.csv（1行1区間：session,kind,y,t0_s,rate_cps,f_out_hz,f_m_hz）、
  split.json（{"train": [学習の正常セッション], "test": [残り]}）、info.json
"""
import argparse
import json
import pathlib
import sys

import numpy as np
import pandas as pd

from ch5lib import (CONDS, FS_DS, KINDS, SEG, encoder_rate, kind_of, list_sessions, load_run, onset_sample,
                    pico_offset, run_end_s, run_start_sample, to_16k)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="data/sessions")
    ap.add_argument("--cpr-out", type=float, required=True)
    ap.add_argument("--pulses", type=int, default=None)
    ap.add_argument("--align", choices=["clock", "onset"], default="clock")
    ap.add_argument("--skip-s", type=float, default=1.0, help="回り始めてから捨てる時間 [s]")
    ap.add_argument("--n-train", type=int, default=6)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="data/ch5_dataset")
    a = ap.parse_args()
    sessions = list_sessions(a.root)
    normal = [s for s in sessions if kind_of(s) == "N"]
    if len(normal) < a.n_train + 2:
        sys.exit(f"正常のセッションが {len(normal)} 個しかありません（学習 {a.n_train} 個＋評価 2 個以上が必要）")
    rng = np.random.default_rng(a.seed)
    train = sorted(rng.choice(normal, a.n_train, replace=False).tolist())
    split = {"train": train, "test": [s for s in sessions if s not in train]}
    print(f"学習（正常）：{', '.join(train)}\n評価：{', '.join(split['test'])}")

    for cond in CONDS:
        xs, rows = [], []
        print(f"\n===== {cond} =====")
        print("セッション 種類   区間 出力軸[Hz] モータ軸[Hz] USB遅れ中央/99%[ms] 立ち上がり−時計[ms]")
        for sname in sessions:
            sdir = pathlib.Path(a.root) / sname
            if not (sdir / f"{cond}.wav").exists():
                continue
            x, fs, enc, meta = load_run(sdir, cond)
            if meta.get("stopped") or meta.get("undervolt") or meta.get("overflow") or meta.get("dropped"):
                print(f"  ⚠ {sname} {cond}：途中で止まった（STOP・I,undervolt）・取りこぼし（録音・Pico B）があるので使いません（録り直す）")
                continue
            _, lag_med, lag99 = pico_offset(enc)
            n0 = run_start_sample(meta, enc, fs)
            on = onset_sample(x, fs, n0)
            diff_ms = (on - n0) / fs * 1000
            if a.align == "onset":
                if np.isnan(on):
                    sys.exit(f"{sname} {cond}：音の立ち上がりが見つかりません（--align clock を使う）")
                n0 = on
            elif not abs(diff_ms) < 30:
                print(f"  ⚠ {sname} {cond}：時計と音の立ち上がりが {diff_ms:.0f} ms ずれています（--align onset も試す）")
            y16 = to_16k(x, fs)
            n0_16 = n0 * FS_DS / fs
            t_end = run_end_s(enc, meta) - 0.05                   # 指令が出ていた最後の直前まで
            starts = np.arange(n0_16 + a.skip_s * FS_DS, n0_16 + t_end * FS_DS - SEG, SEG).astype(int)
            for st in starts:
                t0 = (st - n0_16) / FS_DS                          # この区間の始まり（RUN の開始からの秒）
                rate = encoder_rate(enc, t0 * 1000, (t0 + SEG / FS_DS) * 1000)
                xs.append(y16[st:st + SEG].astype(np.float32))
                rows.append({"session": sname, "kind": kind_of(sname), "y": int(kind_of(sname) != "N"),
                             "t0_s": round(t0, 4), "rate_cps": rate, "f_out_hz": rate / a.cpr_out,
                             "f_m_hz": rate / (4 * a.pulses) if a.pulses else np.nan})
            seg = [r for r in rows if r["session"] == sname]
            fo = np.mean([r["f_out_hz"] for r in seg])
            fm = np.mean([r["f_m_hz"] for r in seg])
            print(f"{sname:>8} {KINDS[kind_of(sname)]:<6} {len(seg):>4} {fo:>9.3f} {fm:>11.1f}"
                  f" {lag_med:>9.1f}/{lag99:<6.1f} {diff_ms:>14.1f}")
        if not xs:
            continue
        out = pathlib.Path(a.out) / cond
        out.mkdir(parents=True, exist_ok=True)
        df = pd.DataFrame(rows)
        np.save(out / "x.npy", np.stack(xs))
        np.save(out / "y.npy", df["y"].to_numpy(np.int64))
        np.save(out / "session.npy", df["session"].to_numpy().astype("U8"))
        df.to_csv(out / "segments.csv", index=False)
        (out / "split.json").write_text(json.dumps(split, ensure_ascii=False, indent=1))
        info = {"cond": cond, "fs": FS_DS, "seg": SEG, "cpr_out": a.cpr_out, "pulses": a.pulses,
                "align": a.align, "skip_s": a.skip_s, "root": str(a.root), "n": len(df),
                "n_by_kind": df["kind"].value_counts().to_dict()}
        (out / "info.json").write_text(json.dumps(info, ensure_ascii=False, indent=1))
        print(f"{out}：{len(df)} 区間（{info['n_by_kind']}）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lines.py
#!/usr/bin/env python3
"""第5章：1つの RUN のスペクトルの線を、エンコーダの回転数から計算した周波数と照らし合わせる

使い方（~/home-lab/ch5 で）:
  uv run python lines.py data/sessions/N01 --cond S2 --cpr-out 2744 --pulses 7 --out out/lines_N01_S2.png
定常の部分（回り始めてから 1 秒後〜止める直前）を、ウェルチ法（2 秒の hann 窓、半分ずつ重ねる。0.5 Hz きざみ）
で平均したパワースペクトルにし、雑音の床（まわり 100 Hz の中央値）より 10 dB 以上高い山を拾う（f_m の 3% より近い山は1つにまとめる）。
山ごとに「モータ軸の周波数 f_m の何倍か」「PWM の周波数の何倍か」を表示し、
いちばん強いモータの線（PWM の線をのぞく）の両側（± 出力軸の周波数 f_out）の側帯波の大きさも表示する（アンバランスの目印）。
S2 では、PWM の線（Pico の時計で 1000 Hz）の周波数を細かく測り、Mac の録音の時計と Pico の時計の比を出す。
"""
import argparse
import pathlib

import numpy as np
from scipy.ndimage import median_filter
from scipy.signal import find_peaks, welch

from ch5lib import encoder_rate, jp_plt, load_run, run_end_s, run_start_sample


def fine_freq(x, fs, f_guess, span=3.0):
    """hann 窓・8 倍のゼロ詰めの FFT の山（f_guess ± span）を、対数振幅の放物線で細かく求める"""
    nfft = 8 * (1 << int(np.ceil(np.log2(len(x)))))
    X = np.abs(np.fft.rfft((x - x.mean()) * np.hanning(len(x)), nfft))
    fr = np.arange(len(X)) * fs / nfft
    sel = np.flatnonzero(np.abs(fr - f_guess) <= span)
    i = sel[np.argmax(X[sel])]
    a, b, c = np.log(X[i - 1:i + 2])
    return fr[i] + 0.5 * (a - c) / (a - 2 * b + c) * fs / nfft


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sdir")
    ap.add_argument("--cond", default="S2")
    ap.add_argument("--cpr-out", type=float, required=True)
    ap.add_argument("--pulses", type=int, required=True)
    ap.add_argument("--fmax", type=float, default=8000.0)
    ap.add_argument("--top", type=int, default=15)
    ap.add_argument("--out", default=None)
    a = ap.parse_args()
    x, fs, enc, meta = load_run(a.sdir, a.cond)
    n0 = run_start_sample(meta, enc, fs)
    t1 = run_end_s(enc, meta) - 0.05
    seg = x[int(n0 + 1.0 * fs):int(n0 + t1 * fs)]
    rate = encoder_rate(enc, 1000, t1 * 1000)
    f_out = rate / a.cpr_out
    f_m = rate / (4 * a.pulses)
    f_pwm = meta["freq_actual"] if meta["d1"] < 1000 else None
    print(f"{a.sdir} {a.cond}：出力軸 {60 * f_out:.1f} RPM（f_out = {f_out:.3f} Hz）、"
          f"モータ軸 f_m = {f_m:.2f} Hz（G = {a.cpr_out / (4 * a.pulses):.2f}）"
          + (f"、PWM {f_pwm:.2f} Hz" if f_pwm else "、PWM なし（デューティ 100%）"))

    fr, P = welch(seg, fs, window="hann", nperseg=2 * fs, noverlap=fs)
    L = 10 * np.log10(P + 1e-30)
    floor = median_filter(L, size=int(100 / (fr[1] - fr[0])) | 1, mode="nearest")
    keep = fr <= a.fmax
    pk, _ = find_peaks(L * keep, height=floor + 10, distance=max(1, int(0.03 * f_m / (fr[1] - fr[0]))))   # f_m の 3% より近い山は1つに
    pk = pk[np.argsort(L[pk] - floor[pk])[::-1][:a.top]]
    print("  周波数[Hz]  床より[dB]   f_m の倍数        PWM の倍数")
    for i in sorted(pk, key=lambda j: fr[j]):
        k = fr[i] / f_m
        s = f"{fr[i]:>10.1f} {L[i] - floor[i]:>9.1f}   {k:>6.2f}（{round(k):>2}倍との差 {fr[i] - round(k) * f_m:+6.1f} Hz）"
        if f_pwm:
            m = fr[i] / f_pwm
            s += f"  {m:>5.2f}" + ("  ← PWM" if abs(m - round(m)) < 0.003 and round(m) > 0 else "")
        print(s)
    mot = [i for i in pk if not (f_pwm and abs(fr[i] / f_pwm - round(fr[i] / f_pwm)) < 0.003)]
    top = mot[int(np.argmax(L[mot]))] if mot else None             # PWM の線をのぞいた、いちばん強い線
    if top is not None:
        def level_at(f):
            return L[np.argmin(np.abs(fr - f))]
        lo, hi = level_at(fr[top] - f_out), level_at(fr[top] + f_out)
        print(f"  いちばん強いモータの線 {fr[top]:.1f} Hz の側帯波（± f_out = {f_out:.2f} Hz）："
              f"下 {lo - L[top]:+.1f} dB、上 {hi - L[top]:+.1f} dB（大きいほど、出力軸1回転ごとの振幅のゆれが大きい）")
    if f_pwm:
        f_meas = fine_freq(seg, fs, f_pwm)
        ppm = (f_meas / f_pwm - 1) * 1e6
        print(f"  PWM の線：{f_meas:.4f} Hz（Pico の時計で {f_pwm:.2f} Hz）→ 比 {ppm:+.1f} ppm。"
              f"Mac の録音の時計が Pico より約 {-ppm:+.1f} ppm 速い（10 秒で {-ppm * 0.01:+.2f} ms のずれ）")
    if a.out:
        plt = jp_plt()
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(fr[keep], L[keep], lw=0.6)
        ax.plot(fr[keep], floor[keep], lw=0.6, color="0.6")
        for k in range(1, int(a.fmax / f_m) + 1):
            ax.axvline(k * f_m, color="C1", lw=0.5, ls=":")
        if f_pwm:
            for m in range(1, int(a.fmax / f_pwm) + 1):
                ax.axvline(m * f_pwm, color="C2", lw=0.6, ls="--")
        ax.plot(fr[pk], L[pk], "rv", ms=4)
        ax.set_xlabel("周波数 [Hz]（橙の点線：f_m の整数倍、緑の破線：PWM の整数倍）")
        ax.set_ylabel("パワー [dB]")
        fig.tight_layout()
        pathlib.Path(a.out).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(a.out, dpi=130)
        print(f"{a.out} を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tf_compare.py
#!/usr/bin/env python3
"""第5章：同じ録音を STFT（短い窓・長い窓）・CWT（複素モルレー）・対数メルスペクトログラムで並べて比べる

使い方（~/home-lab/ch5 で）:
  uv run python tf_compare.py data/sessions/U01 --cond S1 --pulses 7 --out out/tf_U01_S1.png
  --t0 -0.2 --dur 1.5：RUN の開始から見た切り出しの位置と長さ [s]（回り始めと定常の両方が入る）
  --check-pywt：自作の cwt_morlet と PyWavelets の pywt.cwt（cmor）を比べる
エンコーダから求めたモータ軸の周波数 f_m(t) とその6倍を、STFT と CWT の上に白い線で重ねる。
"""
import argparse

import numpy as np

from ch5lib import FS_DS, cwt_morlet, jp_plt, load_run, logmel, run_start_sample, to_16k


def stft_db(y, n_fft, hop):
    import librosa
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop, window="hann", center=True)) ** 2
    return 10 * np.log10(S + 1e-12)


def resolution_table(freqs=(400.0, 4000.0), n_cycles=6.0):
    print("方法                       時間の広がり        周波数の広がり（ガウスの σ・hann の 3dB 幅）")
    for L in (256, 4096):
        print(f"STFT hann {L:>5} 点       {1000 * L / FS_DS:>6.1f} ms（窓の長さ）  "
              f"{1.44 * FS_DS / L:>7.1f} Hz（どの周波数でも同じ）")
    for f in freqs:
        st, sf_ = n_cycles / (2 * np.pi * f), f / n_cycles
        print(f"CWT n_cycles={n_cycles:.0f} @ {f:>5.0f} Hz  σ_t = {1000 * st:>5.2f} ms        σ_f = {sf_:>6.1f} Hz"
              f"（σ_t·σ_f = 1/(2π) = {st * sf_:.4f}）")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sdir")
    ap.add_argument("--cond", default="S1")
    ap.add_argument("--t0", type=float, default=-0.2)
    ap.add_argument("--dur", type=float, default=1.5)
    ap.add_argument("--n-cycles", type=float, default=6.0)
    ap.add_argument("--pulses", type=int, required=True, help="P（第4章で求めた、モータ軸1回転の A 相のパルス数）")
    ap.add_argument("--check-pywt", action="store_true")
    ap.add_argument("--out", default="out/tf.png")
    a = ap.parse_args()
    x, fs, enc, meta = load_run(a.sdir, a.cond)
    y = to_16k(x, fs)
    n0 = run_start_sample(meta, enc, fs) * FS_DS / fs
    i0 = int(n0 + a.t0 * FS_DS)
    seg = y[i0:i0 + int(a.dur * FS_DS)]
    t_rel = a.t0 + np.arange(len(seg)) / FS_DS
    resolution_table(n_cycles=a.n_cycles)

    # エンコーダの f_m(t)：10 ms ごとのカウントの差 ÷ 4P
    pulses = a.pulses
    t_ms = enc["t_ms"].to_numpy()
    c = enc["count"].to_numpy(float)
    tt = np.arange(0, t_ms.max() - 10, 10)
    f_m = (np.interp(tt + 10, t_ms, c) - np.interp(tt, t_ms, c)) / 0.010 / (4 * pulses)

    freqs = np.geomspace(100, 7000, 72)
    W = cwt_morlet(seg, FS_DS, freqs, n_cycles=a.n_cycles, hop=16)
    panels = [("(a) STFT 短い窓（256 点 = 16 ms）", stft_db(seg, 256, 32), 256, 32),
              ("(b) STFT 長い窓（4096 点 = 256 ms）", stft_db(seg, 4096, 256), 4096, 256)]
    plt = jp_plt()
    fig, ax = plt.subplots(4, 1, figsize=(10, 12))
    for k, (title, S, L, hop) in enumerate(panels):
        fr = np.arange(S.shape[0]) * FS_DS / L
        tt_s = a.t0 + np.arange(S.shape[1]) * hop / FS_DS
        ax[k].pcolormesh(tt_s, fr, S, shading="auto", vmin=S.max() - 70, vmax=S.max(), cmap="magma")
        ax[k].set_ylim(0, 7000)
        ax[k].set_title(title, fontsize=10)
    ax[2].pcolormesh(t_rel[::16][:W.shape[1]], freqs, 20 * np.log10(W + 1e-9), shading="auto",
                     vmin=20 * np.log10(W.max()) - 60, cmap="magma")
    ax[2].set_yscale("log")
    ax[2].set_ylim(freqs[0], freqs[-1])
    ax[2].set_title(f"(c) CWT 複素モルレー（n_cycles = {a.n_cycles:g}、縦軸は対数）", fontsize=10)
    M = logmel(seg, FS_DS)
    ax[3].imshow(M, origin="lower", aspect="auto", cmap="magma", vmin=M.max() - 70,
                 extent=[a.t0, a.t0 + a.dur, 0, M.shape[0]])
    ax[3].set_title("(d) 対数メルスペクトログラム（64 帯域、n_fft 1024・hop 256）", fontsize=10)
    ax[3].set_ylabel("メルの帯域の番号")
    for k in (0, 1, 2):
        for h in (1, 6):
            ax[k].plot(tt / 1000, h * f_m, color="w", lw=0.6, ls="--")
        ax[k].set_xlim(a.t0, a.t0 + a.dur)
        ax[k].set_ylabel("周波数 [Hz]")
    ax[2].set_ylim(freqs[0], freqs[-1])
    ax[3].set_xlabel("RUN の開始からの時間 [s]")
    fig.tight_layout()
    fig.savefig(a.out, dpi=120)
    print(f"{a.out} を保存しました")

    if a.check_pywt:
        import pywt
        B = 2 * (a.n_cycles / (2 * np.pi)) ** 2               # cmorB-1.0 の σ_t が n_cycles/(2πf) になる B
        part = seg[:int(0.5 * FS_DS)]
        f_chk = np.array([300.0, 1000.0, 3000.0])
        scales = 1.0 * FS_DS / f_chk                           # cmor の中心周波数 C = 1.0 → f = C·fs/s
        Wp, fp = pywt.cwt(part, scales, f"cmor{B:.4f}-1.0", sampling_period=1 / FS_DS, method="fft")
        Wm = cwt_morlet(part, FS_DS, f_chk, n_cycles=a.n_cycles)
        for i, f in enumerate(f_chk):
            r = np.corrcoef(np.abs(Wp[i]), Wm[i])[0, 1]
            print(f"  pywt と比べる：{f:.0f} Hz（pywt の周波数 {fp[i]:.1f} Hz）の大きさの相関 {r:.4f}、"
                  f"大きさの比の中央値 {np.median(np.abs(Wp[i]) / (Wm[i] + 1e-12)):.3f}（正規化の違い。形が同じなら相関は 1 に近い）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile detect.py
#!/usr/bin/env python3
"""第5章：正常の録音だけで学習する異常検知（ガウス分布＋マハラノビス距離、オートエンコーダ）と ROC-AUC

使い方（~/home-lab/ch5 で）:
  uv run python detect.py data/ch5_dataset/S1 --out out/detect_S1                 # ガウスだけ（数秒）
  uv run python detect.py data/ch5_dataset/S1 --ae --seeds 0 1 2 --out out/detect_S1   # AE も（MPS で数十秒）
特徴：区間（4096 点）ごとの対数メルスペクトログラム（64 帯域 × 17 フレーム、n_fft 1024・hop 256）。
  ガウス：64 帯域ごとのフレームの平均と標準偏差を並べた 128 次元。平均と共分散（Ledoit–Wolf の縮小推定）を
          学習の正常だけで求め、マハラノビス距離を異常の得点にする。
  AE    ：5 フレーム（320 次元）を入力にする全結合のオートエンコーダ。再構成の二乗誤差の区間平均が得点。
分け方：(1) セッション単位（split.json）、(2) 誤り：正常の区間をセッションを無視してでたらめに分ける（リーク）。
区間：(a) 区間を独立とみなすブートストラップ、(b) セッションを単位にするブートストラップ。
"""
import argparse
import json
import pathlib
import time

import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf

from ch5lib import FS_DS, KINDS, boot_auc, jp_plt, logmel, roc_auc, roc_curve


def gauss_fit(F):
    lw = LedoitWolf().fit(F)
    return lw.location_, lw.precision_, lw.shrinkage_


def gauss_score(F, model):
    mu, prec, _ = model
    d = F - mu
    return np.sqrt(np.einsum("ij,jk,ik->i", d, prec, d))       # マハラノビス距離


def ctx_frames(M, c=5):
    """(N, 64, T) → (N, T−c+1, 64·c)：となりあう c フレームを1つの入力にする"""
    N, B, T = M.shape
    return np.stack([M[:, :, t:t + c].transpose(0, 2, 1).reshape(N, B * c) for t in range(T - c + 1)], axis=1)


def pick_device(name):
    import torch
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def ae_scores(Xtr, Xall, seed, dev, epochs=40, batch=256, lr=1e-3):
    """Xtr：(n, W, D) の学習の窓、Xall：(N, W, D) の得点を出す窓。返り値：(N,) の区間ごとの平均二乗誤差"""
    import torch
    import torch.nn as nn
    torch.manual_seed(seed)
    D = Xtr.shape[2]
    mu, sd = Xtr.reshape(-1, D).mean(0), Xtr.reshape(-1, D).std(0) + 1e-6
    tr = torch.tensor(((Xtr - mu) / sd).reshape(-1, D), dtype=torch.float32, device=dev)
    model = nn.Sequential(nn.Linear(D, 128), nn.ReLU(), nn.Linear(128, 32), nn.ReLU(),
                          nn.Linear(32, 128), nn.ReLU(), nn.Linear(128, D)).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator(device="cpu").manual_seed(seed)
    for ep in range(epochs):
        perm = torch.randperm(len(tr), generator=g).to(dev)
        for i in range(0, len(tr) - batch + 1, batch):
            xb = tr[perm[i:i + batch]]
            loss = ((model(xb) - xb) ** 2).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        xa = torch.tensor((Xall - mu) / sd, dtype=torch.float32, device=dev)
        err = ((model(xa) - xa) ** 2).mean(dim=2).mean(dim=1)   # 窓ごと → 区間ごとの平均
    return err.cpu().numpy(), float(loss.item())


def evaluate(score, y, sess, kind, test, label, res):
    """評価の区間（test）だけで、全部の異常と種類ごとの AUC、区間単位・セッション単位の区間を出す"""
    s0, g0 = score[test & (y == 0)], sess[test & (y == 0)]
    out = {}
    for k in ["all"] + [k for k in KINDS if k != "N"]:
        m1 = test & (y == 1) & ((kind == k) if k != "all" else True)
        if m1.sum() == 0:
            continue
        s1, g1 = score[m1], sess[m1]
        auc = roc_auc(s0, s1)
        seg_ci = boot_auc(s0, s1)
        ses_ci = boot_auc(s0, s1, g0, g1)
        out[k] = {"auc": auc, "ci_segment": seg_ci, "ci_session": ses_ci,
                  "n0": int(len(s0)), "n1": int(len(s1)), "sessions0": int(len(set(g0))), "sessions1": int(len(set(g1)))}
        name = "異常すべて" if k == "all" else KINDS[k]
        print(f"  {label:<16} {name:<8} AUC {auc:.3f}  区間単位［{seg_ci[0]:.3f}, {seg_ci[1]:.3f}］"
              f"  セッション単位［{ses_ci[0]:.3f}, {ses_ci[1]:.3f}］（正常 {len(set(g0))}・異常 {len(set(g1))} セッション）")
    res[label] = out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folder")
    ap.add_argument("--ae", action="store_true")
    ap.add_argument("--seeds", type=int, nargs="+", default=[0])
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--fpr-target", type=float, default=0.05, help="しきい値を決める偽陽性率（学習の中の1つ抜き）")
    ap.add_argument("--out", default="out/detect")
    a = ap.parse_args()
    d = pathlib.Path(a.folder)
    X = np.load(d / "x.npy")
    y = np.load(d / "y.npy")
    sess = np.load(d / "session.npy")
    kind = np.array([s[0] for s in sess])
    split = json.loads((d / "split.json").read_text())
    train = np.isin(sess, split["train"])
    test = ~train
    print(f"{d}：{len(X)} 区間、学習の正常 {train.sum()} 区間（{len(split['train'])} セッション）")

    t0 = time.perf_counter()
    M = logmel(X.astype(float), FS_DS)                           # (N, 64, 17)
    F = np.concatenate([M.mean(axis=2), M.std(axis=2)], axis=1)  # (N, 128)
    print(f"対数メルスペクトログラム {M.shape}：{time.perf_counter() - t0:.1f} 秒")
    res = {"folder": str(d), "split": split, "n_train": int(train.sum())}

    # ---- (1) ガウス＋マハラノビス、セッション単位で分ける ----
    model = gauss_fit(F[train])
    print(f"Ledoit–Wolf の縮小の強さ {model[2]:.3f}（128 次元を {train.sum()} 区間で推定）")
    sc_g = gauss_score(F, model)
    print("ガウス＋マハラノビス距離")
    evaluate(sc_g, y, sess, kind, test, "gauss_session", res)

    # しきい値：学習のセッションを1つずつ抜いて当てはめ、抜いたセッションの得点の (1−fpr) 点（評価データは見ない）
    loso = []
    for s in split["train"]:
        m = train & (sess != s)
        loso.append(gauss_score(F[sess == s], gauss_fit(F[m])))
    print("  学習のセッションを1つ抜いたときの、抜いたセッションの得点の中央値："
          + "、".join(f"{s} {np.median(v):.1f}" for s, v in zip(split["train"], loso)))
    thr = float(np.quantile(np.concatenate(loso), 1 - a.fpr_target))
    fpr = float(np.mean(sc_g[test & (y == 0)] > thr))
    tpr = {KINDS[k]: float(np.mean(sc_g[test & (kind == k)] > thr)) for k in KINDS if k != "N"}
    inbag = float(np.quantile(sc_g[train], 1 - a.fpr_target))
    print(f"  しきい値 {thr:.2f}（学習の中の1つ抜きの {100 * (1 - a.fpr_target):.0f}% 点。学習データそのものの"
          f" {100 * (1 - a.fpr_target):.0f}% 点なら {inbag:.2f}）")
    print(f"  評価の正常の偽陽性率 {fpr:.3f}、真陽性率 " + "、".join(f"{k} {v:.3f}" for k, v in tpr.items()))
    res["gauss_threshold"] = {"thr": thr, "thr_inbag": inbag, "fpr_test": fpr, "tpr_test": tpr}

    # ---- (2) 誤り：正常の区間をでたらめに分ける（同じセッションの区間が学習と評価の両方に入る）----
    rng = np.random.default_rng(0)
    idx0 = np.flatnonzero(y == 0)
    tr_leak = np.zeros(len(y), bool)
    tr_leak[rng.choice(idx0, int(train.sum()), replace=False)] = True
    test_leak = ~tr_leak
    sc_leak = gauss_score(F, gauss_fit(F[tr_leak]))
    evaluate(sc_leak, y, sess, kind, test_leak, "gauss_leak", res)

    # ---- (3) オートエンコーダ ----
    sc_ae = None
    if a.ae:
        dev = pick_device(a.device)
        W = ctx_frames(M)                                       # (N, 13, 320)
        aucs = []
        for seed in a.seeds:
            t0 = time.perf_counter()
            sc, last = ae_scores(W[train], W, seed, dev, epochs=a.epochs)
            sec = time.perf_counter() - t0
            print(f"オートエンコーダ（種 {seed}、{dev}、{a.epochs} エポック）：{sec:.1f} 秒、最後の損失 {last:.3f}")
            evaluate(sc, y, sess, kind, test, f"ae_seed{seed}", res)
            res[f"ae_seed{seed}"]["train_s"] = sec
            aucs.append(res[f"ae_seed{seed}"]["all"]["auc"])
            sc_ae = sc if sc_ae is None else sc_ae
        print(f"  AE の AUC（異常すべて）の種ごとの値：{', '.join(f'{v:.3f}' for v in aucs)}")

    out = pathlib.Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"session": sess, "kind": kind, "y": y, "train": train, "gauss": sc_g, "gauss_leak": sc_leak,
                  **({"ae": sc_ae} if sc_ae is not None else {})}).to_csv(out / "scores.csv", index=False)
    (out / "result.json").write_text(json.dumps(res, ensure_ascii=False, indent=1))

    plt = jp_plt()
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    curves = [("ガウス（セッションで分けた）", sc_g, test), ("ガウス（リークあり）", sc_leak, test_leak)]
    if sc_ae is not None:
        curves.append(("AE（セッションで分けた）", sc_ae, test))
    for name, sc, te in curves:
        fp, tp = roc_curve(sc[te & (y == 0)], sc[te & (y == 1)])
        ax[0].plot(fp, tp, label=f"{name} AUC {roc_auc(sc[te & (y == 0)], sc[te & (y == 1)]):.3f}")
    ax[0].plot([0, 1], [0, 1], "k:", lw=0.8)
    ax[0].set_xlabel("偽陽性率")
    ax[0].set_ylabel("真陽性率")
    ax[0].legend(fontsize=8)
    order = sorted(set(sess), key=lambda s: (s[0] != "N", s))
    ax[1].boxplot([sc_g[sess == s] for s in order], tick_labels=order)
    ax[1].axhline(thr, color="r", lw=0.8, ls="--", label="しきい値")
    for i, s in enumerate(order):
        if s in split["train"]:
            ax[1].get_xticklabels()[i].set_color("gray")
    ax[1].set_ylabel("マハラノビス距離")
    ax[1].tick_params(axis="x", rotation=90, labelsize=7)
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out / "detect.png", dpi=130)
    print(f"{out}/detect.png・scores.csv・result.json を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile rpm_audio.py
#!/usr/bin/env python3
"""第5章：音からモータ軸の回転の周波数を推定し、エンコーダの値とブランド・アルトマン図で比べる

使い方（~/home-lab/ch5 で）:
  uv run python rpm_audio.py --root data/sessions --cpr-out 2744 --pulses 7 --out out/rpm
  --pulses を省くと、「エンコーダのカウントの速さ ÷ 音の周波数 = 4P」から P を推定してから比べる
  --fmin 100 --fmax 800：探すモータ軸の周波数の範囲 [Hz]（第4章の結果から決める）、--win 1.0：窓 [s]
1 RUN（10 秒）を 1 秒の窓に分け、窓ごとに harmonic_f0（倍音の和）で音の f_m、同じ時間のカウントの傾きで
エンコーダの f_m を出す。ブランド・アルトマン図は RUN ごとの平均（1 RUN = 1点）で描き、窓を全部独立と
みなした誤った区間も並べて表示する。一致の限界は RUN の平均どうしのものと、1 秒の窓1回どうしのもの
（RUN の間と中の分散を合わせる。Bland & Altman 2007）の両方を、偏りの区間はセッションの平均でも出す。
"""
import argparse
import json
import pathlib

import numpy as np
import pandas as pd
from scipy import stats

from ch5lib import CONDS, FS_DS, encoder_rate, harmonic_f0, jp_plt, kind_of, list_sessions, load_run, \
    run_end_s, run_start_sample, to_16k


def bland_altman(a, b):
    """a − b の平均（偏り）とその t 区間、一致の限界（偏り ± 1.96 sd）とその近似の区間（Bland & Altman 1986）"""
    d = np.asarray(a) - np.asarray(b)
    n = len(d)
    m, sd = d.mean(), d.std(ddof=1)
    t = stats.t.ppf(0.975, n - 1)
    se_m, se_loa = sd / np.sqrt(n), np.sqrt(3 * sd ** 2 / n)
    return {"n": n, "bias": m, "bias_ci": (m - t * se_m, m + t * se_m), "sd": sd,
            "loa": (m - 1.96 * sd, m + 1.96 * sd), "loa_halfwidth_ci": t * se_loa}


def per_run_icc(ok):
    """差（音 − エンコーダ）の一元配置の級内相関 ICC(1)。0 に近ければ窓はほぼ独立、1 に近ければ RUN が単位"""
    d = ok.assign(diff=ok["f_audio"] - ok["f_enc"]).groupby(["session", "cond"])["diff"]
    k = d.size().mean()
    msb = (d.size() * (d.mean() - ok["f_audio"].sub(ok["f_enc"]).mean()) ** 2).sum() / (d.ngroups - 1)
    msw = d.var(ddof=1).mul(d.size() - 1).sum() / (d.size().sum() - d.ngroups)
    return float(max(0.0, (msb - msw) / (msb + (k - 1) * msw)))


def window_loa(ok, bias):
    """1 秒の窓1回の推定どうしの一致の限界（Bland & Altman 2007 の反復測定、真の値が RUN ごとに違う場合）。
    差の分散を RUN の間 σ_b² と RUN の中 σ_w² に分け（一元配置の分散分析）、σ_b² + σ_w² の平方根を使う"""
    g = ok.assign(diff=ok["f_audio"] - ok["f_enc"]).groupby(["session", "cond"])["diff"]
    m, n, N = g.size(), g.ngroups, g.size().sum()
    msb = (m * (g.mean() - ok["f_audio"].sub(ok["f_enc"]).mean()) ** 2).sum() / (n - 1)
    msw = g.var(ddof=1).mul(m - 1).sum() / (N - n)
    m0 = (N - (m ** 2).sum() / N) / (n - 1)                    # 窓の数がそろわないときの「1 RUN の窓の数」
    s_b2 = max(0.0, (msb - msw) / m0)
    sd = float(np.sqrt(s_b2 + msw))
    return {"sd_between": float(np.sqrt(s_b2)), "sd_within": float(np.sqrt(msw)), "sd_total": sd,
            "loa": (bias - 1.96 * sd, bias + 1.96 * sd)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="data/sessions")
    ap.add_argument("--cpr-out", type=float, required=True)
    ap.add_argument("--pulses", type=int, default=None)
    ap.add_argument("--fmin", type=float, default=100.0)
    ap.add_argument("--fmax", type=float, default=800.0)
    ap.add_argument("--win", type=float, default=1.0)
    ap.add_argument("--out", default="out/rpm")
    a = ap.parse_args()
    rows = []
    for sname in list_sessions(a.root):
        for cond in CONDS:
            sdir = pathlib.Path(a.root) / sname
            if not (sdir / f"{cond}.wav").exists():
                continue
            x, fs, enc, meta = load_run(sdir, cond)
            if meta.get("stopped") or meta.get("undervolt") or meta.get("overflow") or meta.get("dropped"):
                print(f"  ⚠ {sname} {cond}：途中で止まった・取りこぼしがあるので使いません")
                continue
            y = to_16k(x, fs)
            n0 = run_start_sample(meta, enc, fs) * FS_DS / fs
            for t0 in np.arange(1.0, run_end_s(enc, meta) - a.win, a.win):
                seg = y[int(n0 + t0 * FS_DS):int(n0 + (t0 + a.win) * FS_DS)]
                f_a, _, sharp = harmonic_f0(seg, FS_DS, a.fmin, a.fmax)
                rate = encoder_rate(enc, t0 * 1000, (t0 + a.win) * 1000)
                rows.append({"session": sname, "kind": kind_of(sname), "cond": cond, "t0_s": t0,
                             "f_audio": f_a, "sharp": sharp, "rate_cps": rate})
    df = pd.DataFrame(rows)
    ratio = df["rate_cps"] / df["f_audio"]                      # = 4P（音が正しく f_m を拾っていれば）
    if a.pulses is None:
        p_est = float(np.median(ratio) / 4)
        print(f"カウントの速さ ÷ 音の周波数 の中央値 = {4 * p_est:.2f} → P ≈ {p_est:.2f}（整数に丸めて {round(p_est)}）")
        pulses = round(p_est)
    else:
        pulses = a.pulses
    print(f"P = {pulses}、G = N_out/(4P) = {a.cpr_out / (4 * pulses):.2f}")
    df["f_enc"] = df["rate_cps"] / (4 * pulses)
    df["r"] = df["f_audio"] / df["f_enc"]
    gross = (df["r"] - 1).abs() > 0.05                         # 5% より外れたもの＝倍数・分数の取り違え
    print(f"窓 {len(df)} 個のうち、音の推定が 5% より外れた窓 {gross.sum()} 個"
          + (f"（比の例：{', '.join(f'{v:.2f}' for v in df.loc[gross, 'r'].head(5))}）" if gross.any() else ""))
    ok = df[~gross]
    per_run = ok.groupby(["session", "cond"]).agg(f_audio=("f_audio", "mean"), f_enc=("f_enc", "mean"),
                                                  n=("f_audio", "size")).reset_index()
    ba = bland_altman(per_run["f_audio"], per_run["f_enc"])
    ba_w = bland_altman(ok["f_audio"], ok["f_enc"])
    per_ses = per_run.assign(d=per_run["f_audio"] - per_run["f_enc"]).groupby("session")["d"].mean()
    ba_s = bland_altman(per_ses.to_numpy(), np.zeros(len(per_ses)))    # セッションの平均（独立な単位）で偏りの区間
    lw = window_loa(ok, ba["bias"])
    print(f"\nブランド・アルトマン（音 − エンコーダ、RUN ごとの平均、n = {ba['n']}。"
          f"S1・S2 は同じセッションなので、独立な単位はセッション {per_run['session'].nunique()} 個）")
    print(f"  偏り {ba['bias']:+.3f} Hz［95% 区間 {ba['bias_ci'][0]:+.3f}, {ba['bias_ci'][1]:+.3f}］、"
          f"一致の限界 {ba['loa'][0]:+.3f}〜{ba['loa'][1]:+.3f} Hz（限界の区間の半幅 ±{ba['loa_halfwidth_ci']:.3f}）")
    print(f"  ※ この一致の限界は「RUN の平均（{per_run['n'].mean() * a.win:.0f} 秒ぶんの窓の平均）どうし」の一致")
    print(f"  偏りの区間をセッションの平均（{ba_s['n']} 点）で出すと［{ba_s['bias_ci'][0]:+.3f}, {ba_s['bias_ci'][1]:+.3f}］")
    print(f"  1 秒の窓1回どうしの一致の限界（RUN の間 sd {lw['sd_between']:.3f}・RUN の中 sd {lw['sd_within']:.3f} を合わせた"
          f" sd {lw['sd_total']:.3f}）：{lw['loa'][0]:+.3f}〜{lw['loa'][1]:+.3f} Hz")
    print(f"  誤り：窓 {ba_w['n']} 個を独立とみなすと 偏りの区間［{ba_w['bias_ci'][0]:+.3f}, {ba_w['bias_ci'][1]:+.3f}］"
          f"（n を {ba_w['n']} と数えている。RUN の中の窓どうしが似ているほど、この区間は狭すぎる）")
    icc = per_run_icc(ok)
    print(f"  窓の差のばらつきのうち RUN の違いによる割合（級内相関）≈ {icc:.2f}")
    rel = 100 * (per_run["f_audio"] / per_run["f_enc"] - 1)
    print(f"  相対の差：平均 {rel.mean():+.3f}%、標準偏差 {rel.std(ddof=1):.3f}%")

    out = pathlib.Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    df.to_csv(out / "rpm_windows.csv", index=False)
    per_run.to_csv(out / "rpm_runs.csv", index=False)
    (out / "rpm_ba.json").write_text(json.dumps({"pulses": pulses, "per_run": ba, "per_window_wrong": ba_w,
                                                 "per_session_bias": ba_s, "window_loa": lw,
                                                 "gross": int(gross.sum())}, ensure_ascii=False, indent=1))
    plt = jp_plt()
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ax[0].scatter(df["f_enc"], df["f_audio"], s=6, c=np.where(gross, "r", "C0"))
    lim = [df["f_enc"].min() * 0.95, df["f_enc"].max() * 1.05]
    ax[0].plot(lim, lim, "k:", lw=0.8)
    ax[0].set_xlabel("エンコーダの f_m [Hz]")
    ax[0].set_ylabel("音の f_m [Hz]")
    mean = (ok["f_audio"] + ok["f_enc"]) / 2
    ax[1].scatter(mean, ok["f_audio"] - ok["f_enc"], s=4, c="0.7", label="1 秒の窓")
    ax[1].scatter((per_run["f_audio"] + per_run["f_enc"]) / 2, per_run["f_audio"] - per_run["f_enc"], s=18,
                  c="C0", label="RUN の平均")
    for v, ls in [(ba["bias"], "-"), (ba["loa"][0], "--"), (ba["loa"][1], "--")]:
        ax[1].axhline(v, color="C3", ls=ls, lw=0.9)
    ax[1].set_xlabel("2つの平均 [Hz]")
    ax[1].set_ylabel("音 − エンコーダ [Hz]")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out / "rpm_ba.png", dpi=130)
    print(f"{out}/rpm_ba.png・rpm_windows.csv・rpm_runs.csv・rpm_ba.json を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile session_ci.py
#!/usr/bin/env python3
"""第5章：得点のファイル（score・y・session の npz）から、AUC と2通りの 95% 区間を出す

使い方（~/home-lab/ch5 で）:
  uv run python session_ci.py ~/home-cloud/runs/p3c6/ae_rec_mac_seed0_scores.npz
III-6 の anomaly_ae.py（第5章の変更点(5)を当てたもの）が保存した得点に、detect.py と同じ
区間単位・セッション単位のブートストラップを当てる。
"""
import sys

import numpy as np

from ch5lib import boot_auc, roc_auc

for path in sys.argv[1:]:
    d = np.load(path)
    s, y, g = d["score"], d["y"], d["session"]
    s0, s1, g0, g1 = s[y == 0], s[y == 1], g[y == 0], g[y == 1]
    a = roc_auc(s0, s1)
    lo, hi = boot_auc(s0, s1)
    slo, shi = boot_auc(s0, s1, g0, g1)
    print(f"{path}：AUC {a:.3f}  区間単位［{lo:.3f}, {hi:.3f}］  セッション単位［{slo:.3f}, {shi:.3f}］"
          f"（正常 {len(set(g0))}・異常 {len(set(g1))} セッション）")

In [ ]:
import glob, os, re, shlex, shutil, subprocess, sys
from IPython.display import Audio, Image, display


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    words = []
    for w in shlex.split(args):
        words += (sorted(glob.glob(w)) or [w]) if any(c in w for c in "*?[") else [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True)
    print(p.stdout, end="")
    err = "\n".join(l for l in p.stderr.splitlines() if "findfont" not in l)   # 候補のうち見つからないフォントの警告は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
    if err.strip():
        print(err, file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=760):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=f, width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i))


def near(a, b, tol):
    return abs(a - b) <= tol + 1e-9


def auc_line(text, label, name):
    """detect.py の表示の1行 → (AUC, 区間単位の下・上, セッション単位の下・上)"""
    m = re.search(rf"^  {re.escape(label)}\s+{re.escape(name)}\s+AUC ([\d.]+)  区間単位［([\d.]+), ([\d.]+)］"
                  rf"  セッション単位［([\d.]+), ([\d.]+)］", text, re.M)
    assert m, f"表示に見つからない：{label} {name}"
    return tuple(float(v) for v in m.groups())


for d in ["data_synth", "out_synth"]:   # やり直したとき：前の結果を消してから作り直す
    shutil.rmtree(d, ignore_errors=True)
print("準備ができました")

## 1. 合成の「録音のセッション」を作る（5-6 の `synth_sound.py`、手順 A の 1）

`setup_ch5.sh` の最後の3行のうちの1行目です（執筆時の Mac で約6秒）。`rec_session.py record` が作るのと同じ形のファイル（`<セッション>/S1.wav`・`S1_enc.csv`・`S1.json`、S2 も）を、正常 N01〜N10、アンバランス U01〜U03、摩擦 F01〜F03、電圧低下 V01〜V03 の 19 セッションぶん作ります（全部で約 70 MB）。

合成の中身は仮の値です：$P = 7$、$G = 98$（$N_{\mathrm{out}} = 2744$）、電池 5.0〜5.4V、Mac の録音の時計が Pico より 30 ppm 速い、録音の最初の標本の時刻に 3 ms 前後の誤差、など（5-6 の `synth_sound.py` の説明）。

In [ ]:
out_syn = run("synth_sound.py --root data_synth/sessions --seed 0")

In [ ]:
# 確かめ（手順 A の 1）
names = sorted(os.listdir("data_synth/sessions"))
ok(len(names) == 19 and sum(n[0] == "N" for n in names) == 10 and all(sum(n[0] == k for n in names) == 3 for k in "UFV"),
   "19 セッション（正常 10、アンバランス・摩擦・電圧低下 3 ずつ）ができた")
ok(all(os.path.exists(f"data_synth/sessions/{n}/{c}{e}") for n in names for c in ("S1", "S2") for e in (".wav", "_enc.csv", ".json")),
   "どのセッションにも S1・S2 の wav・_enc.csv・json の6つのファイルがある")
peaks = [float(v) for v in re.findall(r"最大 (-[\d.]+) dBFS", out_syn)]
ok(len(peaks) == 19 and all(-30 <= p <= -6 for p in peaks), f"最大の大きさが −30〜−6 dBFS の間（{min(peaks)}〜{max(peaks)} dBFS。手順 C の 5 の目安）")

### 合成のモータ音を聞く

N01（正常）・U01（アンバランス）・F01（摩擦）・V01（電圧低下）の S1 を、RUN の開始から 2〜5 秒の 3 秒だけ、16 kHz にして鳴らします。⚠ **先に音量を下げてください。** 録ったままの大きさ（`normalize=False`）なので、最大 −20〜−30 dBFS の小さな音です。音量を少しずつ上げて聞きます（−6 dBFS を超える音は、このセルは鳴らしません）。

- アンバランスは、出力軸1回転（約 0.2 秒）ごとに音の大きさがゆれます（5-3-8）。
- 摩擦は、こすれる「シャー」という音が足され、少し低くなります。電圧低下は、低く・小さくなります。
- 合成の音なので、実物のモータの音とは違います。

In [ ]:
import numpy as np
from ch5lib import load_run, run_start_sample, to_16k


def listen(session, cond="S1", t0=2.0, dur=3.0, max_dbfs=-6.0):
    """RUN の開始から t0 秒のところを dur 秒だけ、録ったままの大きさで鳴らす（大きすぎる音は鳴らさない）"""
    x, fs, enc, meta = load_run(f"data_synth/sessions/{session}", cond)
    n0 = run_start_sample(meta, enc, fs)
    seg = to_16k(x[int(n0 + t0 * fs):int(n0 + (t0 + dur) * fs)], fs)
    peak = 20 * np.log10(np.max(np.abs(seg)) + 1e-12)
    rms = 20 * np.log10(np.sqrt(np.mean(seg ** 2)) + 1e-12)
    print(f"{session} {cond}（{meta['kind']}）：最大 {peak:.1f} dBFS、RMS {rms:.1f} dBFS")
    if peak > max_dbfs:
        print(f"  ⚠ 最大が {max_dbfs} dBFS を超えるので鳴らしません")
        return None
    display(Audio(seg, rate=16000, normalize=False))
    return peak


heard = [listen(s) for s in ["N01", "U01", "F01", "V01"]]
ok(all(p is not None and p < -6 for p in heard), "4つとも −6 dBFS より小さく、録ったままの大きさで鳴らした")

## 2. 時刻をそろえて切り、データセットにする（5-3-3・5-6 の `ch5_dataset.py`、手順 A の 1・E の 9）

Pico の時刻 `t_ms` と Mac の時刻（`time.monotonic`）のずれを「いちばん遅れの小さい行」から求め（最小遅延の推定）、RUN の開始が録音の何番目の標本かを決めます。回り始めの 1 秒を捨て、16 kHz にして 4096 点（0.256 秒）ずつ切ります。学習に使う正常の 6 セッションを `--seed 0` で選び、**セッション単位で**学習と評価に分けます（`split.json`）。

表の「USB遅れ中央/99%」は Pico から Mac への受け取りの遅れ、「立ち上がり−時計」は、時計から求めた RUN の開始と、音の立ち上がりの差です（合成では 0〜10 ms）。

In [ ]:
out_ds = run("ch5_dataset.py --root data_synth/sessions --cpr-out 2744 --pulses 7 --out data_synth/ch5_dataset")

In [ ]:
# 確かめ（手順 A の 1・E の 9 と 5-7 の「時刻合わせ」）
ROW = re.compile(r"^\s+([NUFV]\d\d) \S+\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)/([\d.]+)\s+(-?[\d.]+)$", re.M)
rows = [m.groups() for m in ROW.finditer(out_ds)]
ok("S1：646 区間" in out_ds and "S2：646 区間" in out_ds, "S1・S2 それぞれ「646 区間」")
ok(len(rows) == 38 and all(int(r[1]) == 34 for r in rows), "38 RUN（19 セッション × S1・S2）すべてで「区間 34」")
lag_med = [float(r[4]) for r in rows]
lag99 = [float(r[5]) for r in rows]
diff = [float(r[6]) for r in rows]
ok(all(1.5 <= v <= 3 for v in lag_med) and all(10 <= v <= 30 for v in lag99),
   f"USB の遅れの中央値 約2 ms（{min(lag_med)}〜{max(lag_med)}）、99% 点 {min(lag99)}〜{max(lag99)} ms")
ok(all(-1 <= v <= 10 for v in diff), f"立ち上がり−時計 が 0〜10 ms（{min(diff)}〜{max(diff)} ms。±30 ms 以内なら時計の方法で十分）")
ok("⚠" not in out_ds, "⚠ の行が出ない")
ok("学習（正常）：N01, N03, N04, N05, N07, N09" in out_ds, "学習の正常は N01・N03・N04・N05・N07・N09（--seed 0）")

## 3. スペクトルの線を確かめる（5-3-1・5-6 の `lines.py`、手順 C の 6・E の 10）

定常の部分のパワースペクトル（ウェルチ法）で、雑音の床より 10 dB 以上高い山を拾い、「モータ軸の周波数 $f_m$ の何倍か」「PWM の周波数の何倍か」を表示します。$f_m$ はエンコーダのカウントの速さ ÷ $4P$ です。`run_ch5.sh` は N05 で書いてありますが、ここでは 5-7 の結果の例と比べられるように **N01 の S2**（PWM 70%・1 kHz）で実行します。

In [ ]:
out_l1 = run("lines.py data_synth/sessions/N01 --cond S2 --cpr-out 2744 --pulses 7 --out out_synth/lines_N01_S2.png")
show("out_synth/lines_N01_S2.png")

In [ ]:
# 確かめ（手順 C の 6 と 5-7 の「線」）
peaks = [float(v) for v in re.findall(r"^\s+([\d.]+)\s+[\d.]+\s+[\d.]+（", out_l1, re.M)]
want = [424.0, 849.5, 1270.5, 1697.5, 2118.0, 2541.5]
ok(all(any(near(p, w, 0.5) for p in peaks) for w in want), f"f_m の 1〜6 倍の線 {want} Hz が拾われた")
pwm = [float(v) for v in re.findall(r"^\s+([\d.]+) .*← PWM$", out_l1, re.M)]
ok(any(near(p, 1000, 0.5) for p in pwm) and any(near(p, 2000, 0.5) for p in pwm) and any(near(p, 3000, 0.5) for p in pwm),
   "1000・2000・3000 Hz などに「← PWM」の線がある")
dk = [abs(float(m.group(2))) for m in re.finditer(r"（\s*(\d+)倍との差\s+([-+][\d.]+) Hz）\s+[\d.]+$", out_l1, re.M)]
ok(sum(v <= 3 for v in dk) >= 0.8 * len(dk), f"PWM でない線 {len(dk)} 本のうち {sum(v <= 3 for v in dk)} 本が、f_m の整数倍との差 3 Hz 以内（多くが整数倍に並ぶ）")
f_pwm = num(r"PWM の線：([\d.]+) Hz", out_l1)
ppm = num(r"Mac の録音の時計が Pico より約 \+([\d.]+) ppm 速い", out_l1)
ok(near(f_pwm, 999.9698, 0.0005) and near(ppm, 30.2, 0.3), f"PWM の線 {f_pwm} Hz → Mac の録音の時計が Pico より +{ppm} ppm 速い（入れた値 30 ppm）")

次に、正常（N02）とアンバランス（U01）の S1 で、いちばん強いモータの線の両側（± 出力軸の周波数 $f_{\mathrm{out}}$）の**側帯波**を比べます。アンバランスは出力軸1回転ごとに振幅がゆれるので、側帯波が大きくなります（5-3-8）。

In [ ]:
out_n02 = run("lines.py data_synth/sessions/N02 --cond S1 --cpr-out 2744 --pulses 7 --out out_synth/lines_N02_S1.png")
out_u01 = run("lines.py data_synth/sessions/U01 --cond S1 --cpr-out 2744 --pulses 7 --out out_synth/lines_U01_S1.png")
show("out_synth/lines_U01_S1.png")

In [ ]:
# 確かめ（手順 E の 10 と 5-7：側帯波は N02 で −28／−23 dB、U01 で −17／−15 dB）
sb = lambda t: (num(r"下 ([-+][\d.]+) dB", t), num(r"上 ([-+][\d.]+) dB", t))
n02, u01 = sb(out_n02), sb(out_u01)
ok(all(near(v, w, 0.6) for v, w in zip(n02 + u01, (-28, -23, -17, -15))),
   f"側帯波（下／上）：N02 {n02}、U01 {u01} dB")
ok(min(u01) > max(n02), "U01 の側帯波は、下・上とも N02 より大きい")
ok("PWM なし（デューティ 100%）" in out_u01, "S1（デューティ 100%）では PWM の線がない")

## 4. STFT と CWT を比べる（5-3-4・5-6 の `tf_compare.py`、手順 E の 11）

U01 の S1 の回り始め（RUN の開始の 0.2 秒前から 1.5 秒）を、(a) 短い窓の STFT（256 点 = 16 ms）、(b) 長い窓の STFT（4096 点 = 256 ms）、(c) 複素モルレーの CWT（`ch5lib.py` の自作の `cwt_morlet`、n_cycles = 6）、(d) 対数メルスペクトログラムで並べます。白い破線は、エンコーダから求めた $f_m(t)$ とその 6 倍です。`--check-pywt` で、自作の CWT と PyWavelets の `pywt.cwt`（cmor）を比べます。

In [ ]:
out_tf = run("tf_compare.py data_synth/sessions/U01 --cond S1 --pulses 7 --check-pywt --out out_synth/tf_U01_S1.png")
show("out_synth/tf_U01_S1.png", width=640)

In [ ]:
# 確かめ（手順 E の 11 と 5-7）
ok(re.search(r"STFT hann   256 点 .*  90\.0 Hz", out_tf) and re.search(r"STFT hann  4096 点 .*  5\.6 Hz", out_tf)
   and "@   400 Hz  σ_t =  2.39 ms" in out_tf, "分解能の表：STFT 256 点で 90 Hz、4096 点で 5.6 Hz、CWT の 400 Hz で σ_t = 2.39 ms")
ok(out_tf.count("σ_t·σ_f = 1/(2π) = 0.1592") == 2, "CWT はどの周波数でも σ_t·σ_f = 1/(2π)（不確定性の下限）")
r = [float(v) for v in re.findall(r"大きさの相関 ([\d.]+)", out_tf)]
ok(len(r) == 3 and min(r[1:]) >= 0.99 and r[0] >= 0.97,
   f"pywt との大きさの相関が 1000・3000 Hz で 0.99 以上、300 Hz で 0.97 以上 {r}（教科書は 0.9985〜0.9999。低い周波数は版で 0.98 前後になることがある）")
ratio = [float(v) for v in re.findall(r"大きさの比の中央値 ([\d.]+)", out_tf)]
ok(ratio[0] > ratio[1] > ratio[2], f"大きさの比は周波数で変わる {ratio}（pywt は √スケール で正規化するため）")

## 5. 正常だけで学ぶ異常検知と ROC-AUC（5-3-5・5-3-6・5-6 の `detect.py`、手順 F の 12・13）

区間ごとの対数メルスペクトログラム（64 帯域 × 17 フレーム）から、

- **ガウス＋マハラノビス距離**：64 帯域ごとの平均と標準偏差（128 次元）。学習の正常だけで平均と共分散（Ledoit–Wolf の縮小推定）を求め、マハラノビス距離を得点にする。
- **オートエンコーダ（AE）**：となりあう 5 フレーム（320 次元）を入力にする全結合の AE。再構成の二乗誤差の区間平均を得点にする（`torch`。Mac では MPS、Colab では GPU があれば CUDA、なければ CPU）。

を求め、評価のセッション（正常 4・異常 9）で AUC と 95% 区間を出します。区間は (a) 区間を独立とみなすブートストラップと、(b) セッションを単位にするブートストラップの2通りです。さらに、誤りの例として、正常の区間をセッションを無視してでたらめに分けた「リークのある分け方」の AUC も出します。

In [ ]:
SEEDS, EPOCHS = "0 1 2", 40   # 教科書と同じ設定（執筆時の Mac で約10秒）。重いときは SEEDS = "0"、EPOCHS = 20 などに
out_d1 = run(f"detect.py data_synth/ch5_dataset/S1 --ae --seeds {SEEDS} --epochs {EPOCHS} --out out_synth/detect_S1")
show("out_synth/detect_S1/detect.png")

In [ ]:
# 確かめ（手順 F の 12・13 と 5-7 の「異常検知」「しきい値」。S1）
g_all, g_u = auc_line(out_d1, "gauss_session", "異常すべて"), auc_line(out_d1, "gauss_session", "アンバランス")
ok(near(g_all[0], 0.957, 0.002) and near(g_u[0], 0.871, 0.002),
   f"ガウスの AUC：異常すべて {g_all[0]}（区間単位［{g_all[1]}, {g_all[2]}］、セッション単位［{g_all[3]}, {g_all[4]}］）、アンバランス {g_u[0]}")
ok(near(g_all[1], 0.939, 0.003) and near(g_all[2], 0.972, 0.003) and near(g_all[3], 0.854, 0.01) and near(g_all[4], 1.0, 0.001),
   "区間：区間単位［0.939, 0.972］、セッション単位［0.854, 1.000］（5-7 と同じ）")
ok(g_all[4] - g_all[3] > 2 * (g_all[2] - g_all[1]) and g_u[4] - g_u[3] > 2 * (g_u[2] - g_u[1]),
   "セッション単位の区間は、区間単位の区間よりずっと広い（異常すべて・アンバランスとも2倍以上）")
leak = auc_line(out_d1, "gauss_leak", "異常すべて")[0]
ok(near(leak, 0.997, 0.002) and leak > g_all[0], f"リークのある分け方の AUC {leak} は、セッションで分けたとき（{g_all[0]}）より高い")
ok(all(auc_line(out_d1, "gauss_session", k)[0] == 1.0 for k in ("摩擦", "電圧低下")), "摩擦と電圧低下はガウスで AUC 1.000")
loso = dict((m.group(1), float(m.group(2))) for m in re.finditer(r"(N\d\d) ([\d.]+)", out_d1.split("得点の中央値：")[1].split("\n")[0]))
ok(loso == {"N01": 16.5, "N03": 34.4, "N04": 18.9, "N05": 18.9, "N07": 32.8, "N09": 158.4},
   f"1つ抜きの得点の中央値 {loso}（N09 だけが特に大きい）")
thr, inbag = num(r"しきい値 ([\d.]+)", out_d1), num(r"学習データそのものの 95% 点なら ([\d.]+)", out_d1)
ok(near(thr, 160.34, 0.05) and near(inbag, 9.17, 0.02) and min(loso.values()) > inbag,
   f"しきい値（1つ抜きの 95% 点）{thr}、学習データそのものの 95% 点 {inbag}。1つ抜きの得点は、どれも学習データそのものの 95% 点より大きい")
ok("評価の正常の偽陽性率 0.000、真陽性率 アンバランス 0.000、摩擦 0.000、電圧低下 0.000" in out_d1,
   "このしきい値では、評価の正常も異常も1つも超えない（偽陽性率 0、真陽性率 0）")
ae = [auc_line(out_d1, f"ae_seed{s}", "異常すべて")[0] for s in SEEDS.split()]
ae_u = [auc_line(out_d1, f"ae_seed{s}", "アンバランス")[0] for s in SEEDS.split()]
ok(all(0.85 <= v <= 0.95 for v in ae) and max(ae_u) < g_u[0],
   f"AE の AUC {ae}（教科書は 0.898・0.909・0.894）。アンバランスは {ae_u} で、ガウス（{g_u[0]}）より低い")
if SEEDS == "0 1 2" and EPOCHS == 40:
    ok(all(near(v, w, 0.02) for v, w in zip(ae, [0.898, 0.909, 0.894])),
       "教科書の設定のとき：AE の種ごとの AUC が 5-7 の値（0.898・0.909・0.894）と ±0.02 以内（動かす機器で少し変わる）")

S2（PWM 70%・1 kHz）でも同じように実行します（`run_ch5.sh` の D の2行目）。

In [ ]:
out_d2 = run(f"detect.py data_synth/ch5_dataset/S2 --ae --seeds {SEEDS} --epochs {EPOCHS} --out out_synth/detect_S2")

In [ ]:
# 確かめ（5-7：S2 はガウス 0.945、アンバランス 0.835、リーク 1.000、AE 0.892〜0.902。1つ抜きの 95% 点 139.9 で、電圧低下だけが全部超える）
g2, g2u = auc_line(out_d2, "gauss_session", "異常すべて")[0], auc_line(out_d2, "gauss_session", "アンバランス")[0]
ok(near(g2, 0.945, 0.002) and near(g2u, 0.835, 0.002) and auc_line(out_d2, "gauss_leak", "異常すべて")[0] == 1.0,
   f"S2：ガウス {g2}、アンバランス {g2u}、リーク 1.000")
ok(near(num(r"しきい値 ([\d.]+)", out_d2), 139.87, 0.05) and "摩擦 0.000、電圧低下 1.000" in out_d2,
   "S2：しきい値 139.87 で、電圧低下だけが全部超えた")
ae2 = [auc_line(out_d2, f"ae_seed{s}", "異常すべて")[0] for s in SEEDS.split()]
ok(all(0.85 <= v <= 0.95 for v in ae2), f"S2 の AE の AUC {ae2}（教科書は 0.892〜0.902）")

## 6. 音から回転数を推定する（5-3-7・5-6 の `rpm_audio.py`、手順 F の 14）

1 RUN（10 秒）を 1 秒の窓に分け、窓ごとに「$k \cdot f$（$k$ = 1〜10）の線の強さの和が最大になる $f$」として音のモータ軸の周波数を、同じ時間のカウントの傾きからエンコーダの $f_m$ を出します。まず `--pulses` を省いて、「カウントの速さ ÷ 音の周波数 = $4P$」から $P$ を推定します。

In [ ]:
out_r0 = run("rpm_audio.py --root data_synth/sessions --cpr-out 2744 --out out_synth/rpm_noP")

In [ ]:
# 確かめ（手順 F の 14：4P が整数×4 に近い。合成データでは 28.00 → P = 7）
ok(near(num(r"中央値 = ([\d.]+) → P", out_r0), 28.0, 0.02) and "P = 7、G = N_out/(4P) = 98.00" in out_r0,
   "カウントの速さ ÷ 音の周波数 の中央値 28.00 → P = 7、G = 98.00（synth_sound.py に入れた値と同じ）")

次に、第4章で求めた $P = 7$ を渡して、ブランド・アルトマン図で比べます。点は RUN ごとの平均（38 点）ですが、S1・S2 は同じセッションなので、独立な単位はセッション 19 個です。窓（304 個）を全部独立とみなした区間は、誤りの例として並べて表示されます。

In [ ]:
out_r = run("rpm_audio.py --root data_synth/sessions --cpr-out 2744 --pulses 7 --out out_synth/rpm")
show("out_synth/rpm/rpm_ba.png")

In [ ]:
# 確かめ（手順 F の 14 と 5-7 の「音の回転数」）
ok("窓 304 個のうち、音の推定が 5% より外れた窓 0 個" in out_r, "取り違えた窓（5% より外れた窓）0 個")
bias, lo, hi = (num(r"偏り ([-+][\d.]+) Hz［95% 区間 ([-+][\d.]+), ([-+][\d.]+)］", out_r, i) for i in (1, 2, 3))
ok(near(bias, -0.048, 0.002) and near(lo, -0.110, 0.002) and near(hi, 0.014, 0.002) and lo < 0 < hi,
   f"偏り {bias} Hz［{lo}, {hi}］：区間が 0 を含む")
l1, l2 = num(r"一致の限界 ([-+][\d.]+)〜", out_r), num(r"一致の限界 [-+][\d.]+〜([-+][\d.]+) Hz", out_r)
w1, w2 = num(r"sd [\d.]+）：([-+][\d.]+)〜", out_r), num(r"sd [\d.]+）：[-+][\d.]+〜([-+][\d.]+) Hz", out_r)
ok(near(l1, -0.418, 0.003) and near(l2, 0.323, 0.003), f"一致の限界（RUN の平均どうし）{l1}〜{l2} Hz（約 ±0.1%）")
ok(near(w1, -1.153, 0.005) and near(w2, 1.057, 0.005) and (w2 - w1) > 2.5 * (l2 - l1),
   f"1 秒の窓1回どうしの一致の限界 {w1}〜{w2} Hz は、RUN の平均どうしの約3倍の幅")
s1, s2 = num(r"セッションの平均（19 点）で出すと［([-+][\d.]+),", out_r), num(r"セッションの平均（19 点）で出すと［[-+][\d.]+, ([-+][\d.]+)］", out_r)
ok(near(s1, -0.117, 0.002) and near(s2, 0.021, 0.002), f"セッションの平均（19 点）で出した偏りの区間［{s1}, {s2}］")
ok("窓の差のばらつきのうち RUN の違いによる割合（級内相関）≈ 0.00" in out_r and "偏りの区間［-0.111, +0.015］" in out_r,
   "合成データでは級内相関 0.00 なので、窓を全部使った区間［−0.111, +0.015］とほとんど変わらない")

## 7. III-6 の課題1 の入口（5-6-1 の `motor_rec.py`・`session_ci.py`、手順 G の 15 の一部）

教科書では、`data/ch5_dataset/S1` を `~/home-cloud/data/ch5_motor/` にコピーし、`motor_rec.py` と、変更点を当てた III-6 の `anomaly_ae.py` で学習します。このノートブックには III-6 の `anomaly_ae.py`・`prep_cache.py` が無いので、ここでは2つだけを試します。

1. `motor_rec.py` の `MotorRec` が、`split.json` のとおりに学習 204 区間・評価 442 区間を返すこと（`python motor_rec.py …` の表示の後半、`log_stft` の平均・標準偏差は III-5 の `prep_cache.py` が要るので出しません）。
2. `session_ci.py`：教科書では `anomaly_ae.py` の変更点(5) が保存した得点のファイル（`score`・`y`・`session` の npz）に使います。ここでは、**かわりに 5 節の `detect.py` の AE（種 0）の評価の得点**を同じ形の npz にして渡します。同じ関数（`ch5lib.py` の `boot_auc`、種 0）なので、5 節の `ae_seed0` の「異常すべて」の行と同じ区間が出るはずです。

In [ ]:
%%writefile motor_rec.py
#!/usr/bin/env python3
"""III-6 課題1 で、第5章の録音（ch5_dataset.py の出力）を使うための Dataset（~/home-cloud/motor_rec.py）

  MotorRec(folder, part)  part = "train"（学習の正常セッションだけ）| "test"（残りの正常＋すべての異常）| "all"
      返り値は III-2 の SynthMotor と同じ (x, y)：x は (1, 4096) の float32（16 kHz）、y は 0＝正常・1＝異常。
      SynthMotor と同じく、区間ごとに標準偏差で割る（std_norm=True）。音の大きさの違い（電圧低下で小さくなるなど）は消える。
  uv run python motor_rec.py data/ch5_motor/S1
      数と、学習の正常の log_stft（III-5）の平均・標準偏差を表示する（anomaly_ae.py の features の 1.0・0.7 の代わり）
numpy と torch だけを使う（~/home-cloud に scipy などを足さない）。
"""
import json
import os
import sys

import numpy as np
import torch
from torch.utils.data import Dataset


class MotorRec(Dataset):
    def __init__(self, folder, part="train", std_norm=True):
        x = np.load(os.path.join(folder, "x.npy"), mmap_mode="r")
        y = np.load(os.path.join(folder, "y.npy"))
        sess = np.load(os.path.join(folder, "session.npy"))
        with open(os.path.join(folder, "split.json")) as f:
            split = json.load(f)
        train = np.isin(sess, split["train"])
        keep = {"train": train, "test": ~train, "all": np.ones(len(y), bool)}[part]
        self.idx = np.flatnonzero(keep)
        self.x, self.y, self.session, self.std_norm = x, y, sess, std_norm
        if part == "train" and self.y[self.idx].any():
            raise ValueError("学習に異常の区間が入っています（split.json を確かめる）")

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = np.array(self.x[j], dtype=np.float32)
        if self.std_norm:
            x = x / (x.std() + 1e-8)
        return torch.from_numpy(x).unsqueeze(0), int(self.y[j])


if __name__ == "__main__":
    from prep_cache import log_stft
    folder = sys.argv[1]
    tr, te = MotorRec(folder, "train"), MotorRec(folder, "test")
    print(f"学習 {len(tr)} 区間（正常だけ）、評価 {len(te)} 区間（正常 {int((te.y[te.idx] == 0).sum())}・"
          f"異常 {int((te.y[te.idx] == 1).sum())}）")
    xs = torch.cat([tr[i][0] for i in range(len(tr))])
    s = log_stft(xs)[:, :128, :64]
    print(f"学習の正常の log_stft：平均 {s.mean().item():.3f}、標準偏差 {s.std().item():.3f}"
          "（anomaly_ae.py の features の (s - 1.0) / 0.7 をこの値に直す）")

In [ ]:
from motor_rec import MotorRec
tr, te = MotorRec("data_synth/ch5_dataset/S1", "train"), MotorRec("data_synth/ch5_dataset/S1", "test")
x0, y0 = tr[0]
print(f"学習 {len(tr)} 区間（正常だけ）、評価 {len(te)} 区間（正常 {int((te.y[te.idx] == 0).sum())}・異常 {int((te.y[te.idx] == 1).sum())}）")
ok(len(tr) == 204 and len(te) == 442, "学習 204 区間・評価 442 区間（手順 G の 15 の合成データの場合）")
ok(tuple(x0.shape) == (1, 4096) and y0 == 0 and abs(float(x0.std()) - 1) < 1e-3, "1つの区間は (1, 4096)・正常（y = 0）で、標準偏差 1 に割ってある")

In [ ]:
import pandas as pd
sc = pd.read_csv("out_synth/detect_S1/scores.csv")
t = sc[~sc["train"]]                                           # 評価の区間だけ（anomaly_ae.py の test と同じ並び）
np.savez("ae_detect_seed0_scores.npz", score=t["ae"].to_numpy(), y=t["y"].to_numpy(), session=t["session"].to_numpy().astype("U8"))
out_ci = run("session_ci.py ae_detect_seed0_scores.npz")   # 教科書：session_ci.py ~/home-cloud/runs/p3c6/ae_rec_mac_seed0_scores.npz

In [ ]:
# 確かめ：session_ci.py の区間が、detect.py の ae_seed0（異常すべて）の行と同じ
a, lo, hi, slo, shi = auc_line(out_d1, "ae_seed0", "異常すべて")
ok(f"AUC {a:.3f}  区間単位［{lo:.3f}, {hi:.3f}］  セッション単位［{slo:.3f}, {shi:.3f}］（正常 4・異常 9 セッション）" in out_ci,
   f"session_ci.py：AUC {a:.3f}、区間単位［{lo:.3f}, {hi:.3f}］、セッション単位［{slo:.3f}, {shi:.3f}］（detect.py と同じ）")

## おわりに

- 合成のモータ音で、時刻合わせ（最小遅延の推定、立ち上がりとの差 0〜10 ms）、スペクトルの線と PWM の線（Mac の録音の時計が +30 ppm 速いことまで分かる）、アンバランスの側帯波を確かめました。
- STFT は窓の長さで時間と周波数の分解能が決まり、CWT は周波数ごとに窓の長さが変わること、自作の CWT が PyWavelets と同じ形になることを見ました。
- 異常検知では、**セッション単位で分けること**（リークのある分け方は AUC を高く見せる）と、**セッション単位の区間**（区間を独立とみなす区間は狭すぎる）、しきい値を学習の中の1つ抜きで決めると「見たことのないセッション」に引きずられることを見ました。
- 音から回転数を推定すると、$P$ も音とカウントから決まり、ブランド・アルトマン図では「RUN の平均どうし」と「1 秒の窓どうし」で一致の限界が約3倍違うことを見ました。
- このノートブックに入れていないもの：Mac の内蔵マイクと Pico B での録音（`rec_session.py`、手順 B〜D）、第4章の `motor_run.py` での確認、III-6 の `anomaly_ae.py` での学習。これらは Mac で教科書の手順のとおりに行います。合成データの値は仮の値で、実機ではモータの個体差・部屋・置き方で大きく変わります。